# 01 - Understanding the data

Goal: before cleaning anything, see what is actually in the export. Arjun says refunds are > 1 crore/quarter, helpdesk says ~11 lakh/quarter. Want to find out why.

In [ ]:
import pandas as pd
import numpy as np

pd.set_option('display.max_columns', 30)
pd.set_option('display.max_colwidth', 80)

tickets = pd.read_csv('tickets.csv')
agents = pd.read_csv('agents.csv')
orders = pd.read_csv('orders.csv')
customers = pd.read_csv('customers.csv')
products = pd.read_csv('products.csv')

for name, df in [('tickets', tickets), ('agents', agents), ('orders', orders),
                 ('customers', customers), ('products', products)]:
    print(f'{name:10s} {df.shape}')

In [ ]:
tickets.head(3)

In [ ]:
# missing values per column
tickets.isna().sum().to_frame('nulls').assign(pct=lambda d: (d.nulls / len(tickets) * 100).round(1))

## Basic distributions

In [ ]:
for col in ['status', 'channel', 'category', 'priority', 'assigned_team', 'source_system', 'replacement_issued']:
    print(tickets[col].value_counts(dropna=False), '\n')

In [ ]:
tickets['created_at'] = pd.to_datetime(tickets['created_at'])
print('date range:', tickets.created_at.min(), '->', tickets.created_at.max())

# policy s9: new helpdesk went live 14 Sep 2025, so legacy rows should all be before that
print(tickets.groupby('source_system').created_at.agg(['min', 'max']))

## Duplicates (Sameer: "some tickets appear twice from the migration re-import")

In [ ]:
print('rows:', len(tickets), '| unique ticket_ids:', tickets.ticket_id.nunique())

dup_ids = tickets[tickets.ticket_id.duplicated(keep=False)]
print('duplicated ids:', dup_ids.ticket_id.nunique())

# is every duplicate a helpdesk + legacy_fd pair?
dup_ids.groupby('ticket_id').source_system.apply(lambda s: tuple(sorted(s))).value_counts()

In [ ]:
# which columns actually differ between the two copies?
cols = [c for c in tickets.columns if c not in ('source_system',)]
diff_counts = {}
for tid, g in dup_ids.groupby('ticket_id'):
    a, b = g.iloc[0], g.iloc[1]
    for c in cols:
        if not (a[c] == b[c] or (pd.isna(a[c]) and pd.isna(b[c]))):
            diff_counts[c] = diff_counts.get(c, 0) + 1
diff_counts

## Refund amounts - where does the crore come from?

In [ ]:
refunds = tickets[tickets.refund_amount_inr.notna()]
print('refund rows:', len(refunds))
print('raw total (what Arjun sums): Rs', f'{refunds.refund_amount_inr.sum():,.0f}')

refunds.groupby('source_system').refund_amount_inr.describe()

In [ ]:
# legacy median looks ~100x helpdesk. check on duplicate pairs that have a refund in both copies
pairs = (dup_ids[dup_ids.refund_amount_inr.notna()]
         .pivot_table(index='ticket_id', columns='source_system', values='refund_amount_inr')
         .dropna())
pairs['ratio'] = pairs['legacy_fd'] / pairs['helpdesk']
print(pairs.ratio.value_counts())
pairs.head()

Ratio is exactly 100 -> legacy Freshdesk stored paise (policy s9: "legacy tool stored monetary values in its own native unit").

In [ ]:
# quick what-if: fix paise + drop duplicates (keep helpdesk copy), see if it lands near helpdesk's ~11L/quarter
t = tickets.copy()
t['amount'] = np.where(t.source_system == 'legacy_fd', t.refund_amount_inr / 100, t.refund_amount_inr)
t = t.sort_values('source_system').drop_duplicates('ticket_id', keep='first')  # 'helpdesk' sorts before 'legacy_fd'

print('clean total: Rs', f'{t.amount.sum():,.0f}')
t.groupby(t.created_at.dt.to_period('Q')).amount.sum().map('{:,.0f}'.format)

## Reason codes

In [ ]:
r = t[t.amount.notna()]
r.groupby('refund_reason_code').amount.agg(['count', 'sum']).assign(
    pct_count=lambda d: (d['count'] / d['count'].sum() * 100).round(1)
).sort_values('count', ascending=False)

In [ ]:
# goodwill cap is Rs 500 (policy s5) - how many GW-OTHER refunds are above it?
gw = r[r.refund_reason_code == 'GW-OTHER']
print('GW-OTHER refunds:', len(gw), '| above 500:', (gw.amount > 500).sum())
gw[gw.amount > 500].groupby(gw.created_at.dt.to_period('Q')).amount.agg(['count', 'sum'])

## Refund AND replacement on same ticket (policy says never)

In [ ]:
both = r[r.replacement_issued == 'Y']
print('refund + replacement:', len(both), 'of', len(r), 'refunds')
print(both.groupby(both.created_at.dt.to_period('Q')).size())
both.assigned_team.value_counts()

## Agents

In [ ]:
print('roster rows:', len(agents), '| unique agent_ids:', agents.agent_id.nunique())
print(agents.team.value_counts())

# any agent ids in tickets that aren't on the roster?
missing = set(t.agent_id.dropna()) - set(agents.agent_id)
print('agent ids not in roster:', missing)

In [ ]:
# policy s6: returns desk should do the 'large majority' of refunds. does it?
r_team = r.merge(agents[['agent_id', 'team']], on='agent_id', how='left')
r_team.groupby('team').amount.agg(['count', 'sum']).assign(
    share=lambda d: (d['count'] / d['count'].sum() * 100).round(1)
).sort_values('count', ascending=False)

## Joins to reference data

In [ ]:
print('order_id missing:', t.order_id.isna().mean().round(3))

# customers often type the order id in their message, e.g. 'vr896352'
found = t.loc[t.order_id.isna(), 'customer_message'].str.extract(r'(VR\d{6})', flags=2)[0]  # flags=2 -> ignore case
print('recoverable from message:', found.notna().sum(), 'of', t.order_id.isna().sum())

print('skus not in products:', set(t.product_sku.dropna()) - set(products.sku))
print('customers not in customers.csv:', len(set(t.customer_id) - set(customers.customer_id)))

## Free text - what the AI will have to read

In [ ]:
gw[['ticket_id', 'amount', 'customer_message', 'agent_notes']].sample(6, random_state=1)

In [ ]:
# notes that talk about a refund but refund_amount is blank
no_amt = t[t.amount.isna() & t.agent_notes.str.contains(r'refund|rfnd|refnd', case=False, na=False)]
print(len(no_amt))
no_amt[['ticket_id', 'category', 'agent_notes']].sample(6, random_state=1)

## CSAT check (Priya: "CSAT went up 0.4" after Q4 change)

In [ ]:
# blanks = no response, excluded (policy s8). mean() skips NaN already
t.groupby(t.created_at.dt.to_period('Q')).csat_score.agg(['mean', 'count']).round(2)

## Notes so far

- 12,238 rows but only 11,600 tickets. 638 appear twice (helpdesk + legacy_fd), and the only column that differs is the refund amount.
- Legacy refund = exactly 100x helpdesk on every pair -> Freshdesk stored paise. That's why Arjun sees Rs 23 cr. After fixing both it's ~Rs 67L total, ~12-16L a quarter, close to helpdesk's "11L".
- Refunds really did go up: ~6L (Q1'25) -> ~16L (Q4'25), then ~12.5L.
- GW-OTHER = 42% of refunds, 879 of them above the Rs 500 goodwill cap. Reading samples, many have an obvious real reason in the notes (return QC, payment failed) -> the AI can re-code these.
- 166 tickets have refund + replacement (policy: never). Mostly Chat and Logistics, not Returns Desk.
- Returns Desk does only 26% of refunds, policy says "large majority".
- CSAT flat around 3.5 all 18 months, couldn't find the +0.4.
- 1,192 notes mention a refund but have no amount. Some are probably "refund initiated" elsewhere (auto-refund / PG). Needs checking.
- 34% missing order_id, only 212 recoverable from the message -> need the customer+sku fallback.
- Some notes are useless ("as discussed ~Amit"), so the AI needs an UNCLEAR option.